# Pelajaran 04 - Corak Reka Bentuk Penggunaan Alat

Dalam pelajaran ini anda akan mempelajari corak reka bentuk **Penggunaan Alat** untuk agen AI menggunakan Microsoft Agent Framework (Python). Kami merangkumi:

- Mendefinisikan fungsi alat dengan dekorator `@tool` dan parameter berjenis
- Menyediakan skema alat supaya model tahu apa fungsi setiap alat
- Mengawal pelaksanaan alat dengan `approval_mode`
- Mengembalikan **output berstruktur** melalui model Pydantic dan `response_format`

Senario ini adalah seorang **agen tempahan perjalanan** yang boleh mencari destinasi, memeriksa ketersediaan, dan mendapatkan maklumat penerbangan.


## Persediaan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mendefinisikan Alat dengan Dekorator @tool

Dekorator `@tool` menukar fungsi Python biasa menjadi alat yang boleh dipanggil oleh ejen.
Perkara penting:

- **Docstring** menjadi penerangan alat yang dilihat oleh model.
- **Anotasi jenis** (termasuk `Annotated` dengan penerangan) menentukan skema alat.
- `approval_mode` mengawal sama ada pengguna mesti meluluskan setiap panggilan sebelum ia dilaksanakan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Membuat Ejen dengan Pelbagai Alat

Serahkan ketiga-tiga alat kepada pelanggan supaya model dapat menggunakan mana-mana alat yang diperlukan untuk menjawab soalan pengguna.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Output Berstruktur dengan Alat

Dengan menetapkan `response_format` kepada model Pydantic, ejen dipaksa untuk mengembalikan objek JSON bertip yang baik dan bukan teks bentuk bebas. Ini berguna apabila kod hiliran perlu menggunakan hasil secara programatik.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Corak Kelulusan Alat

Parameter `approval_mode` pada `@tool` mengawal sama ada panggilan alat memerlukan kelulusan manusia sebelum dijalankan:

| Mod | Tingkah laku |
|---|---|
| `"never_require"` | Alat berjalan secara automatik — tiada pengesahan pengguna diperlukan. |
| `"always_require"` | Setiap panggilan mesti diluluskan oleh pengguna sebelum ia dijalankan. |

Gunakan `"always_require"` untuk alat yang mempunyai kesan sampingan (contohnya menempah penerbangan, mengenakan caj kad kredit) supaya manusia kekal dalam kawalan.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Ringkasan

Dalam pelajaran ini anda belajar cara untuk:

1. **Mentakrifkan alat** menggunakan dekorator `@tool` dengan parameter berjenis dan docstrings yang berfungsi sebagai skema alat.
2. **Menggabungkan pelbagai alat** supaya ejen boleh memanggilnya secara berurutan untuk menjawab pertanyaan yang kompleks.
3. **Mengembalikan output berstruktur** dengan menyerahkan model Pydantic sebagai `response_format`.
4. **Mengawal kelulusan alat** dengan `approval_mode` untuk memastikan kehadiran manusia dalam proses bagi operasi sensitif.

Corak ini membentuk asas untuk membina ejen yang boleh dipercayai dan bersedia untuk produksi yang dapat berinteraksi dengan sistem luaran dengan selamat.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
